In [ ]:
import os
import glob
import shutil
import pandas as pd
from pathlib import Path

### Move .yml file to directories

In [ ]:
yaml_path = r"\\allen\aind\scratch\ophys\Andrew\metadata\device.yml"
basepath = r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP7\826031'
data_paths = glob.glob(os.path.join(basepath,'**','VCO1_Behavior.harp'),recursive=True)
data_paths

In [ ]:
for path in data_paths:
    if 'device.yml' in os.listdir(path):
        print(path)
        print('device metadata in path')
    else:
        shutil.copy(yaml_path,path)
        new_path = glob.glob(os.path.join(path,'**.yml'))[0]
        print(new_path)

### Move bonsai_event_log.csv to processing directories

In [ ]:
for path in data_paths:
    if 'bonsai_event_log.csv' in os.listdir(path):
        print(path)
        print('Behavior data in path')
    else:
        try:
            parent = Path(path).parents[0]
            behavior_path  = glob.glob(os.path.join(parent,'bonsai_event_log.csv'))[0]
            shutil.move(behavior_path,path)
            new_path = glob.glob(os.path.join(path,'**.csv'))[0]
            print(new_path)
        except:
            print(path)

### Move instrument.json to session_directories

In [ ]:
basepath = r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
json_path = r"\\allen\aind\scratch\ophys\Andrew\metadata\instrument.json"

In [ ]:
mice = [803496,804730,804733,810196,809043,809044,809047,803121,814591,814593,
        825854,825855,824806,826031,826032,826033,826034,834788,
        838410,836173,836174
]

In [ ]:
from __future__ import annotations

import re
import shutil
from pathlib import Path

def find_session_root_subdir(session_dir: Path, mouse: int | str) -> Path:
    """
    Find the subdirectory inside `session_dir` whose name matches:
        nnnnnn_yyyy-mm-dd_hh-mm-ss
    where nnnnnn == mouse id (6 digits).
    Returns the matching Path, or raises ValueError with a helpful message.
    """
    mouse = str(mouse)
    pattern = re.compile(rf"^{re.escape(mouse)}_\d{{4}}-\d{{2}}-\d{{2}}_\d{{2}}-\d{{2}}-\d{{2}}$")

    matches = [p for p in session_dir.iterdir() if p.is_dir() and pattern.match(p.name)]
    if len(matches) == 0:
        raise ValueError(f"No matching destination subdir in {session_dir}")
    if len(matches) > 1:
        raise ValueError(f"Multiple matching destination subdirs in {session_dir}: {[m.name for m in matches]}")
    return matches[0]

def move_instrument_json(session_dir: Path, mouse: int | str, overwrite: bool = False) -> Path:
    """
    Move session_dir/instrument.json into the matched subdirectory.
    Returns destination path.
    """
    src = session_dir / "instrument.json"
    if not src.exists():
        raise FileNotFoundError(f"Missing {src}")

    dest_dir = find_session_root_subdir(session_dir, mouse)
    dest = dest_dir / "instrument.json"

    if dest.exists():
        if overwrite:
            dest.unlink()
        else:
            raise FileExistsError(f"Destination already has instrument.json: {dest}")

    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(dest))
    return dest

In [ ]:
for mouse in mice:
    mouse = str(mouse)

    # Find the mouse directory robustly (no glob()[0] crashes)
    candidates = list(Path(basepath).rglob(mouse))
    candidates = [p for p in candidates if p.is_dir()]

    if not candidates:
        print(f"[SKIP] mouse {mouse}: no directory found under {basepath}")
        continue
    if len(candidates) > 1:
        # pick the shortest path as a heuristic; adjust if you have a better rule
        candidates = sorted(candidates, key=lambda p: len(str(p)))
        print(f"[WARN] mouse {mouse}: multiple dirs found, using {candidates[0]}")
    data_dir = candidates[0]

    session_folders = [
        p for p in data_dir.iterdir()
        if p.is_dir()
        and "processed_data" not in p.name
        and "predictive_processing_data" not in p.name
    ]

    for session_dir in session_folders:
        src = session_dir / "instrument.json"

        # your original conditional, but safer
        if not src.exists():
            continue
        if (session_dir / "behavior").exists():
            continue

        try:
            dest = move_instrument_json(session_dir, mouse, overwrite=False)
            print(f"[OK] moved {src.name} -> {dest}")
        except Exception as e:
            print(f"[ERROR] {session_dir}: {e}")

In [ ]:
session_folders = []

for mouse in mice:
    data_dir = glob.glob(os.path.join(basepath,'**',str(mouse)))[0]
    for path in os.listdir(data_dir):
        session_folder = os.path.join(data_dir,path)
        if os.path.isdir(session_folder) and 'processed_data' not in session_folder and 'predictive_processing_data' not in session_folder:
            print(session_folder)
            session_folders.append(session_folder)

In [ ]:
for folder in session_folders:
    static_data = glob.glob(os.path.join(Path(folder).parent,'**','**VasMap**'),recursive=True)
    print(static_data)

In [ ]:
for mouse in mice:
    data_dir = glob.glob(os.path.join(basepath,'**',str(mouse)))[0]
    vas_file = glob.glob(os.path.join(data_dir,'**','**local_vasculature**'),recursive=True)
    print(vas_file)

In [ ]:
df = pd.DataFrame(columns = ['session_id',
                             'subject_id',
                             'session_#',
                             'session_date',
                             'indicator1',
                             'indicator2',
                             'dmd1_depth',
                             'dmd2_depth',
                             'paradigm',
                             'session_type',
                             'stimulus',
                             'experimentor',
                             'imaging_rig',
                             'behavior_rig',
                             'session_dir',
                             'purpose',
                             'notes'
])


In [ ]:
df['session_dir'] = session_folders

In [ ]:
df['session_id'] = [Path(path).stem for path in session_folders]

In [ ]:
import re
from datetime import datetime
from typing import Optional, Dict, Any

# Matches:
#   yyyy-mm-dd_nnnnnn
#   nnnnnn_yyyy-mm-dd
#   nnnnnn_yyyy-mm-dd_hh-mm-ss
_PAT = re.compile(
    r"""
    ^(?:
        (?P<date1>\d{4}-\d{2}-\d{2})_(?P<sid1>\d{6})               # yyyy-mm-dd_nnnnnn
        (?:_(?P<big1>Big[A-Za-z]+))?                               # optional _Big...
      |
        (?P<sid2>\d{6})_(?P<date2>\d{4}-\d{2}-\d{2})               # nnnnnn_yyyy-mm-dd ...
        (?:_(?P<time2>\d{2}-\d{2}-\d{2}))?                         # optional _hh-mm-ss
        (?:_(?P<big2>Big[A-Za-z]+))?                               # optional _Big...
    )$
    """,
    re.VERBOSE,
)

def parse_subject_and_date(s: str) -> Dict[str, Any]:
    """
    Parses:
      - yyyy-mm-dd_nnnnnn
      - yyyy-mm-dd_nnnnnn_BigVolume / _BigStacks (or any _Big<letters>)
      - nnnnnn_yyyy-mm-dd
      - nnnnnn_yyyy-mm-dd_hh-mm-ss
      - (optionally) ..._BigSomething at the end

    Returns:
      {
        'subject_id': str (6 digits),
        'date': datetime.date,
        'datetime': datetime | None,   # present if time included
      }
    """
    m = _PAT.match(s.strip())
    if not m:
        raise ValueError(f"Unrecognized format: {s!r}")

    subject_id = m.group("sid1") or m.group("sid2")
    date_str   = m.group("date1") or m.group("date2")
    time_str   = m.group("time2")  # only possible in sid2/date2 branch

    d = datetime.strptime(date_str, "%Y-%m-%d").date()

    dt: Optional[datetime] = None
    if time_str is not None:
        dt = datetime.strptime(f"{date_str}_{time_str}", "%Y-%m-%d_%H-%M-%S")

    return {"subject_id": subject_id, "date": d, "datetime": dt}

In [ ]:
for session in df['session_id'].values:
    parse = parse_subject_and_date(session)
    print(parse['date'])

In [ ]:
filen = '2026-02-16_session_df.csv'
savepath = os.path.join(basepath,filen) 
df.to_csv(savepath)

In [ ]:
Path(path).stem

### Delete old metadata files

In [ ]:
from pathlib import Path

root = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")  # change as needed
dry_run = False  # set to False to actually delete

for path in root.rglob("instrument.json"):
    if dry_run:
        print(f"[DRY RUN] Would delete: {path}")
    else:
        try:
            path.unlink()
            print(f"Deleted: {path}")
        except Exception as e:
            print(f"Failed to delete {path}: {e}")

### Find/delete unneeded data

In [ ]:
basepath = r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics"
summary_path = glob.glob(os.path.join(basepath,'**summary.xlsx'))[0]
summary_df = pd.read_excel(summary_path,sheet_name='subjects')
session_df = pd.read_excel(summary_path,sheet_name='sessions')

In [ ]:
assets_processing = pd.read_excel(summary_path,sheet_name='assets_processing')

In [ ]:
delete_df = assets_processing[assets_processing['preprocessed']=='yes']

In [ ]:
delete_df

In [ ]:
key_match = 'glutamate_analysis'

session_sizes = []

for idx, row in delete_df.iterrows():
    try:
        sess_path = row['session_dir']   # change if your column name is different

        alignment_paths = glob.glob(
            os.path.join(sess_path, '**', key_match),
            recursive=True
        )

        total_bytes = 0
        for path in alignment_paths:
            if os.path.isfile(path):
                total_bytes += os.path.getsize(path)

        total_tb = total_bytes / (1024**4)   # tebibytes-style TB
        total_gb = total_bytes / (1024**3)

        session_sizes.append({
            'session_path': sess_path,
            'n_files_found': len(alignment_paths),
            'total_bytes': total_bytes,
            'total_gb': total_gb,
            'total_tb': total_tb,
        })
    except:
        pass
size_df = pd.DataFrame(session_sizes)
size_df

In [ ]:
import os
import glob
from pathlib import Path
import pandas as pd
from datetime import datetime

key_match = 'glutamate_analysis'
dry_run = False  # set to False to actually delete files

# choose where to save the log
log_dir = Path.cwd() / 'deletion_logs'
log_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
log_csv = log_dir / f'instrument_delete_log_{timestamp}.csv'
summary_csv = log_dir / f'instrument_delete_summary_{timestamp}.csv'

file_log = []
session_summary = []

for idx, row in session_df.iterrows():
    sess_path = row['session_dir']  # change if needed
    try:
        alignment_paths = glob.glob(
            os.path.join(sess_path,'**', key_match),
            recursive=True
        )
    except:
        alignment_paths = glob.glob(
            os.path.join(sess_path,'**', key_match),
            recursive=True
        )  
    
#     matched_files = [p for p in alignment_paths if os.path.isfile(p)]
    matched_files = [p for p in alignment_paths if os.path.isdir(p)]

    print(f'\nSession: {sess_path}')
    print(f'Found {len(matched_files)} matching files')

    deleted_count = 0
    deleted_bytes = 0
    error_count = 0

    for path in matched_files:
        size_bytes = os.path.getsize(path)
        status = None
        error_msg = ''

        if dry_run:
            status = 'would_delete'
            print(f'[DRY RUN] Would delete: {path}')
            deleted_count += 1
            deleted_bytes += size_bytes
        else:
            try:
#                 os.remove(path)
                shutil.rmtree(path)
                status = 'deleted'
                print(f'[DELETED] {path}')
                deleted_count += 1
                deleted_bytes += size_bytes
            except Exception as e:
                status = 'error'
                error_msg = str(e)
                error_count += 1
                print(f'[ERROR] {path}: {e}')

        file_log.append({
            'session_path': sess_path,
            'file_path': path,
            'file_name': Path(path).name,
            'size_bytes': size_bytes,
            'size_gb': size_bytes / (1024**3),
            'size_tb': size_bytes / (1024**4),
            'status': status,
            'error': error_msg,
            'dry_run': dry_run,
        })

    session_summary.append({
        'session_path': sess_path,
        'n_files_matched': len(matched_files),
        'n_files_deleted_or_would_delete': deleted_count,
        'n_errors': error_count,
        'total_size_gb': deleted_bytes / (1024**3),
        'total_size_tb': deleted_bytes / (1024**4),
        'dry_run': dry_run,
    })

# save detailed per-file log
# file_log_df = pd.DataFrame(file_log)
# file_log_df.to_csv(log_csv, index=False)

# save per-session summary
# summary_df = pd.DataFrame(session_summary)
# summary_df.to_csv(summary_csv, index=False)

print(f'\nDetailed log saved to:\n{log_csv}')
print(f'Summary log saved to:\n{summary_csv}')

summary_df